In [52]:
import json
import optuna
import joblib
import pandas as pd
import numpy as np
from pathlib import Path
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit
from credit_risk.evaluation import compute_metrics, tune_threshold, evaluate_model
from credit_risk.features import prep_one_split, DROP_COLS, NUMERICAL_COLS, CATEGORICAL_COLS
from credit_risk.dataset import load_splits, AFTER_EDA
from credit_risk.utils import to_jsonable
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import average_precision_score

In [2]:
DROP_COLS.remove('issue_d')

In [3]:
train_df, val_df, test_df, _ = load_splits(AFTER_EDA)

2026-06-15 12:37:54.408 | INFO     | credit_risk.dataset:load_splits:263 - Checking if the files exists...
2026-06-15 12:37:54.420 | INFO     | credit_risk.dataset:load_splits:265 - Loading the Cached files...
2026-06-15 12:37:54.709 | INFO     | credit_risk.dataset:load_splits:273 - Loaded sucessfully all the splits and the metadata, Train_df shape: (466042, 110), val_df shape: (420204, 110), test_df shape: (431712, 110)


In [4]:
filtered_train_df, y_train = prep_one_split(df=train_df, drop_cols=DROP_COLS)
filtered_val_df, y_val = prep_one_split(df=val_df, drop_cols=DROP_COLS)
filtered_test_df, y_test = prep_one_split(df=test_df, drop_cols=DROP_COLS)

2026-06-15 12:37:55.437 | INFO     | credit_risk.features:prep_one_split:207 - Inside Function: prep_one_split
2026-06-15 12:37:55.437 | INFO     | credit_risk.features:split_target_and_features:143 - Inside Function: split_target_and_features
2026-06-15 12:37:55.437 | INFO     | credit_risk.features:split_target_and_features:144 - Splitting the target and the features...
2026-06-15 12:37:55.440 | INFO     | credit_risk.features:split_target_and_features:147 - features and target are splitted successfully...
2026-06-15 12:37:55.440 | INFO     | credit_risk.features:add_credit_yrs:160 - Inside Function: add_credit_yrs
2026-06-15 12:37:55.440 | INFO     | credit_risk.features:add_credit_yrs:162 - Adding the feature 'credit_yrs'
2026-06-15 12:37:55.454 | INFO     | credit_risk.features:add_credit_yrs:164 - 'credit_age_yrs added successfully!'
2026-06-15 12:37:55.455 | INFO     | credit_risk.features:add_fico_mid:169 - Inside Function: add_fico_mid
2026-06-15 12:37:55.455 | INFO     | cred

In [5]:
'issue_d' in filtered_train_df.columns

True

In [6]:
filtered_train_df['issue_d'].is_monotonic_increasing

False

In [7]:
filtered_val_df['issue_d'].is_monotonic_increasing

False

In [8]:
filtered_test_df['issue_d'].is_monotonic_increasing

False

In [9]:
min(filtered_train_df['issue_d']), max(filtered_train_df['issue_d'])

(Timestamp('2007-06-01 00:00:00'), Timestamp('2014-12-01 00:00:00'))

In [10]:
min(filtered_test_df['issue_d']), max(filtered_test_df['issue_d'])

(Timestamp('2016-01-01 00:00:00'), Timestamp('2016-12-01 00:00:00'))

In [11]:
min(filtered_val_df['issue_d']), max(filtered_val_df['issue_d'])

(Timestamp('2015-01-01 00:00:00'), Timestamp('2015-12-01 00:00:00'))

In [12]:
(filtered_train_df.index == y_train.index).all()

np.True_

In [13]:
issue_d_train = filtered_train_df['issue_d']

In [14]:
sort_idx = issue_d_train.sort_values().index
filtered_train_df = filtered_train_df.loc[sort_idx]
y_train = y_train.loc[sort_idx]
issue_d_train = issue_d_train.loc[sort_idx]

In [23]:
issue_d_val = filtered_val_df['issue_d']
sort_idx = issue_d_val.sort_values().index
filtered_val_df = filtered_val_df.loc[sort_idx]
y_val = y_val.loc[sort_idx]
issue_d_val = issue_d_val.loc[sort_idx]

In [24]:
issue_d_test = filtered_test_df['issue_d']
sort_idx = issue_d_test.sort_values().index
filtered_test_df = filtered_test_df.loc[sort_idx]
y_test = y_test.loc[sort_idx]
issue_d_test = issue_d_test.loc[sort_idx]

In [25]:
filtered_val_df['issue_d'].is_monotonic_increasing and filtered_test_df['issue_d'].is_monotonic_increasing

True

In [15]:
issue_d_train.is_monotonic_increasing

True

In [16]:
filtered_train_df['issue_d'].is_monotonic_increasing

True

In [20]:
num_pipeline = Pipeline([
    ('inpute', SimpleImputer(strategy='median')),
    ('scalar', StandardScaler())
])

cat_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

In [21]:
preprocessor = ColumnTransformer([
    ('num', num_pipeline, NUMERICAL_COLS),
    ('cat', cat_pipeline,CATEGORICAL_COLS),
], remainder='drop')

In [22]:
preprocessor.fit(filtered_train_df)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

In [26]:
X_train_feat = preprocessor.transform(filtered_train_df)
X_val_feat = preprocessor.transform(filtered_val_df)
X_test_feat = preprocessor.transform(filtered_test_df)

In [27]:
tscv = TimeSeriesSplit(n_splits=5)

In [29]:
for i, (train_idx, val_idx) in enumerate(tscv.split(X_test_feat)):
    train_dates = issue_d_train.iloc[train_idx]
    val_dates = issue_d_train.iloc[val_idx]
    train_y = y_train.iloc[train_idx]
    val_y = y_train.iloc[val_idx]
    
    print(f"Fold {i}:")
    print(f"  train: {train_dates.min().date()} → {train_dates.max().date()} "
          f"| {len(train_idx):,} rows | {train_y.sum():,} defaults")
    print(f"  val:   {val_dates.min().date()} → {val_dates.max().date()} "
          f"| {len(val_idx):,} rows | {val_y.sum():,} defaults")

Fold 0:
  train: 2007-06-01 → 2012-09-01 | 71,952 rows | 11,357 defaults
  val:   2012-09-01 → 2013-06-01 | 71,952 rows | 11,217 defaults
Fold 1:
  train: 2007-06-01 → 2013-06-01 | 143,904 rows | 22,574 defaults
  val:   2013-06-01 → 2013-12-01 | 71,952 rows | 11,258 defaults
Fold 2:
  train: 2007-06-01 → 2013-12-01 | 215,856 rows | 33,832 defaults
  val:   2013-12-01 → 2014-04-01 | 71,952 rows | 11,790 defaults
Fold 3:
  train: 2007-06-01 → 2014-04-01 | 287,808 rows | 45,622 defaults
  val:   2014-04-01 → 2014-07-01 | 71,952 rows | 12,808 defaults
Fold 4:
  train: 2007-06-01 → 2014-07-01 | 359,760 rows | 58,430 defaults
  val:   2014-07-01 → 2014-11-01 | 71,952 rows | 13,121 defaults


In [36]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'n_jobs': -1,
        'eval_metric': 'logloss',
        'verbosity': 0,
    }
    
    scores = []
    
    for train_idx, val_idx in tscv.split(X_train_feat):
        X_fold_train = X_train_feat[train_idx]
        y_fold_train = y_train.iloc[train_idx]
        X_fold_val = X_train_feat[val_idx]
        y_fold_val = y_train.iloc[val_idx]
        
        model = XGBClassifier(**params)
        model.fit(X_fold_train, y_fold_train.to_numpy())
        
        val_proba = model.predict_proba(X_fold_val)[:, 1]
        score = average_precision_score(y_fold_val.to_numpy(), val_proba)
        
        scores.append(score)
        
    return np.mean(scores)

In [37]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

[I 2026-06-15 13:30:12,484] A new study created in memory with name: no-name-ed704eb7-0163-45a7-b0a0-876995c44db6


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-06-15 13:30:39,918] Trial 0 finished with value: 0.30633183063786285 and parameters: {'n_estimators': 745, 'max_depth': 6, 'learning_rate': 0.04218655100461362, 'subsample': 0.571800056421794, 'colsample_bytree': 0.7815512785918184, 'min_child_weight': 6, 'reg_alpha': 0.00010953887693828094, 'reg_lambda': 0.3736080949973227}. Best is trial 0 with value: 0.30633183063786285.
[I 2026-06-15 13:31:07,390] Trial 1 finished with value: 0.30938101334058704 and parameters: {'n_estimators': 589, 'max_depth': 7, 'learning_rate': 0.03662438560249844, 'subsample': 0.8971311494398087, 'colsample_bytree': 0.7929356869971939, 'min_child_weight': 10, 'reg_alpha': 3.7407110508603107e-06, 'reg_lambda': 6.83748264459472e-08}. Best is trial 1 with value: 0.30938101334058704.
[I 2026-06-15 13:31:38,579] Trial 2 finished with value: 0.30230745151928196 and parameters: {'n_estimators': 876, 'max_depth': 5, 'learning_rate': 0.054036840190493686, 'subsample': 0.517712476569176, 'colsample_bytree': 0.76

In [38]:
print("Best PR-AUC:", study.best_value)
print("Best params:", study.best_params)

Best PR-AUC: 0.3134494619597944
Best params: {'n_estimators': 895, 'max_depth': 7, 'learning_rate': 0.011425456812654167, 'subsample': 0.5833351412852172, 'colsample_bytree': 0.7668598048854729, 'min_child_weight': 10, 'reg_alpha': 1.0261238904989621e-05, 'reg_lambda': 0.0006454118557372465}


In [39]:
tuned_params = study.best_params
tuned_xgb_model = XGBClassifier(**tuned_params)
tuned_xgb_model.fit(X_train_feat, y_train.to_numpy())

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.7668598048854729
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import 

In [41]:
y_train_proba = tuned_xgb_model.predict_proba(X_train_feat)[:, 1]
y_val_proba = tuned_xgb_model.predict_proba(X_val_feat)[:, 1]
y_test_proba = tuned_xgb_model.predict_proba(X_test_feat)[:, 1]

In [42]:
tuned_xgb_eval = evaluate_model(
    y_train=y_train.to_numpy(),
    train_proba=y_train_proba,
    y_val=y_val.to_numpy(),
    val_proba=y_val_proba,
    y_test=y_test.to_numpy(),
    test_proba=y_test_proba,
    fn_cost=10000,
    fp_cost=2000
)

In [43]:
tuned_xgb_eval

{'threshold': np.float64(0.16),
 'train': {'ROC-AUC': 0.7439300157154369,
  'PR-AUC': 0.381689819454631,
  'brier_score': 0.12316741794347763,
  'precision': 0.2783728747025197,
  'recall': 0.7311166956633803,
  'confusion_matrix': array([[241383, 147064],
         [ 20864,  56731]])},
 'val': {'ROC-AUC': 0.708335490425068,
  'PR-AUC': 0.34885062650079623,
  'brier_score': 0.13714288175106049,
  'precision': 0.2789071369106183,
  'recall': 0.7163796119902025,
  'confusion_matrix': array([[200124, 142917],
         [ 21885,  55278]])},
 'test': {'ROC-AUC': 0.6935841329758616,
  'PR-AUC': 0.3104578352376564,
  'brier_score': 0.13049693405628204,
  'precision': 0.2591251104930686,
  'recall': 0.6604027583175361,
  'confusion_matrix': array([[221458, 137456],
         [ 24722,  48076]])}}

In [47]:
model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', tuned_xgb_model)
])

In [45]:
cwd = Path.cwd()
project_root = cwd.parent
project_root

PosixPath('/Users/ak007/SML/Credit-Risk-Default-Prediction-System')

In [50]:
# saving the tuned model
model_path = project_root / 'models' / 'tuned_xgb'
model_path.mkdir(parents=True, exist_ok=True)

In [51]:
joblib.dump(model_pipeline, filename=model_path / 'model.pkl')
joblib.dump(study, filename=model_path / 'optuna_study.pkl')

['/Users/ak007/SML/Credit-Risk-Default-Prediction-System/models/tuned_xgb/optuna_study.pkl']

In [53]:
with open(model_path / 'metric.json', 'w') as f:
    json.dump(to_jsonable(tuned_xgb_eval), f, indent=2)
    
with open(model_path / 'best_params.json', 'w') as f:
    json.dump(tuned_params, f, indent=2)